# MIDI to Piano-Roll
Loads files listed in `lmd_full_metadata.csv` and converts each MIDI to a fixed-length piano-roll for CNN training.

Filters applied:
- `initial_bpm` ∈ [30, 240]
- `n_notes` ∈ [50, 50 000]
- `duration_s` ∈ [15, 600]

Outputs:
- `data/processed_data/piano_roll/` — uint8 `.npy` files, shape `(128, 468)`
  - 128 pitch bins × 468 frames (first 15 s at 31.25 fps, raw velocity 0–127)
  - ~4× smaller than float32; normalize at load time: `roll.astype(np.float32) / 127.0`

Load convention: `roll = np.load("file.npy")` — uint8, shape `(128, 468)`

In [9]:
import numpy as np
import pandas as pd
import pretty_midi
import os
from pathlib import Path
from tqdm.auto import tqdm

## 1. Load metadata

In [10]:
SCRIPTS_DIR = Path(os.path.abspath(''))
PROJECT_DIR = SCRIPTS_DIR.parent.parent
METADATA_CSV = PROJECT_DIR / 'data' / 'processed_data' / 'lmd_full_metadata.csv'
RAW_DATA_DIR = PROJECT_DIR / 'data' / 'raw_data' / 'lmd_full'
OUT_DIR      = PROJECT_DIR / 'data' / 'processed_data' / 'piano_roll'

OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(METADATA_CSV, parse_dates=False)
print(f"Loaded {len(df):,} files from metadata")
df.head(3)

Loaded 155,681 files from metadata


,file,filepath,duration_s,resolution,n_instruments,n_notes,n_tempo_changes,initial_bpm,n_key_changes,initial_key,n_time_sig_changes,initial_time_sig,key_name
0,d66652ff61cdea1eda265490600e6e40.mid,../data/raw_data/lmd_full/d/d66652ff61cdea1eda...,176.59,384,11,5244,1,118.0,0,NaN,1,4/4,NaN
1,d1e757857fb459edf3e543c3c8a79c6a.mid,../data/raw_data/lmd_full/d/d1e757857fb459edf3...,219.15,192,10,4974,1,72.0,0,NaN,1,4/4,NaN
2,d597590e6423c93180f56f2854bfb109.mid,../data/raw_data/lmd_full/d/d597590e6423c93180...,176.97,384,5,3965,62,120.0,13,0.0,2,1/4,C major


## 2. Configuration

In [11]:
PIANO_ROLL_FS = 31.25    # frames per second (must match model input)
CLIP_FRAMES   = 468      # 15 s × 31.25 fps = 468.75 → truncate to 468

# Set N_SAMPLES=None to process ALL files.
N_SAMPLES     = None     # process all files that pass the filters

# Number of parallel worker processes (CPU-bound). Reduce if you hit memory pressure.
N_WORKERS     = os.cpu_count()

## 3. Conversion helpers

In [12]:
def midi_to_piano_roll(filepath: str, fs: float = PIANO_ROLL_FS) -> np.ndarray:
    """Return a (128, CLIP_FRAMES) piano-roll array (uint8, velocity 0–127).

    Uses the first 15 seconds only (CLIP_FRAMES = 468 at 31.25 fps).
    Normalize at load time: roll.astype(np.float32) / 127.0
    """
    midi = pretty_midi.PrettyMIDI(filepath)
    roll = midi.get_piano_roll(fs=fs)          # (128, T), velocity 0-127
    roll = roll[:, :CLIP_FRAMES]               # first 15 s only
    return roll.astype(np.uint8)

## 4. Batch conversion and saving

In [13]:
import warnings

def _process_file(args):
    """Worker: parse one MIDI and save its piano-roll as a .npy file.

    Uses fork-based multiprocessing, so all globals (PIANO_ROLL_FS, CLIP_FRAMES, etc.)
    are inherited from the parent process — no re-import needed.
    Returns None on success, or a dict {'file': ..., 'error': ...} on failure.

    Storage format: uint8, shape (128, 468), raw velocity 0–127 (~60 KB each).
    Normalize at load time: roll.astype(np.float32) / 127.0
    Only the first 15 seconds are kept (CLIP_FRAMES = 468 at 31.25 fps).
    """
    fpath, pr_out_str = args
    pr_out = Path(pr_out_str)
    if pr_out.exists():
        return None  # idempotent — skip already-processed files
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            midi = pretty_midi.PrettyMIDI(fpath)
        roll = midi.get_piano_roll(fs=PIANO_ROLL_FS)   # (128, T), velocity 0-127
        roll = roll[:, :CLIP_FRAMES]                    # first 15 s only → (128, 468)
        roll = roll.astype(np.uint8)                    # raw velocity, 4× smaller than float32
        np.save(pr_out, roll)                           # saves as .npy (uint8, ~60 KB each)
        return None
    except Exception as e:
        return {'file': Path(fpath).name, 'error': str(e)}


In [14]:
import multiprocessing as mp

# ── 1. Sample / filter ────────────────────────────────────────────────────────
df_batch = df.head(N_SAMPLES).reset_index(drop=True) if N_SAMPLES else df.reset_index(drop=True)

before = len(df_batch)
df_batch = df_batch[
    df_batch['initial_bpm'].between(30, 240) &
    df_batch['n_notes'].between(50, 50_000) &
    df_batch['duration_s'].between(15, 600)
].reset_index(drop=True)
print(f"Filters (bpm 30–240, notes 50–50k, duration 15–600s): {before:,} → {len(df_batch):,} files "
      f"({before - len(df_batch):,} removed)")

piano_roll_dir = OUT_DIR
piano_roll_dir.mkdir(parents=True, exist_ok=True)

# ── 2. Pre-filter already-processed files ─────────────────────────────────────
existing     = {p.stem for p in piano_roll_dir.glob('*.npy')}
stems_series = df_batch['file'].str[:-4]           # strip '.mid' suffix — vectorized
todo_mask    = ~stems_series.isin(existing)
n_skip       = int((~todo_mask).sum())
if n_skip:
    print(f"Skipping {n_skip:,} already-processed files.")
df_todo = df_batch[todo_mask].reset_index(drop=True)

# ── 3. Build task list — vectorized ───────────────────────────────────────────
in_paths  = df_todo['filepath'].str.lstrip('../').apply(lambda p: str(PROJECT_DIR / p)).values
stems_arr = df_todo['file'].str[:-4].values
out_paths = np.array([str(piano_roll_dir / f"{s}.npy") for s in stems_arr])

# ── 4. Sort longest-first for load balancing ──────────────────────────────────
sort_idx = df_todo['duration_s'].values.argsort()[::-1]
tasks = list(zip(in_paths[sort_idx], out_paths[sort_idx]))

print(f"Processing {len(tasks):,} files with {N_WORKERS} workers...")

# ── 5. Parallel pool ──────────────────────────────────────────────────────────
chunksize = max(1, min(50, len(tasks) // (N_WORKERS * 20))) if tasks else 1

errors = []
ctx = mp.get_context('fork')
with ctx.Pool(processes=N_WORKERS, maxtasksperchild=1000) as pool:
    for result in tqdm(
        pool.imap_unordered(_process_file, tasks, chunksize=chunksize),
        total=len(tasks),
    ):
        if result is not None:
            errors.append(result)

print(f"\nDone. Errors: {len(errors)}")
if errors:
    err_path = OUT_DIR / 'conversion_errors.csv'
    pd.DataFrame(errors).to_csv(err_path, index=False)
    print(f"Error log saved to {err_path}")

Filters (bpm 30–240, notes 50–50k, duration 15–600s): 155,681 → 150,017 files (5,664 removed)
Skipping 150,014 already-processed files.
Processing 3 files with 14 workers...


  0%|          | 0/3 [00:00<?, ?it/s]


Done. Errors: 0


## 5. Validate and remove corrupted files

In [15]:
def _check_file(fpath):
    """Return (path, reason) if invalid, else None."""
    try:
        arr = np.load(fpath, allow_pickle=False)
        if arr.shape != (128, CLIP_FRAMES):
            return str(fpath), f"bad shape {arr.shape}"
        if arr.dtype != np.uint8:
            return str(fpath), f"bad dtype {arr.dtype}"
        return None
    except Exception as e:
        return str(fpath), str(e)

files = list(OUT_DIR.glob('*.npy'))
print(f"Validating {len(files):,} files...")

ctx = mp.get_context('fork')
with ctx.Pool(processes=N_WORKERS) as pool:
    results = list(tqdm(
        pool.imap_unordered(_check_file, files, chunksize=200),
        total=len(files),
    ))

bad = [r for r in results if r is not None]
print(f"Corrupted/invalid: {len(bad)}")
for p, reason in bad:
    print(f"  {Path(p).name}: {reason}")

for p, _ in bad:
    Path(p).unlink()
if bad:
    print(f"\nRemoved {len(bad)} file(s).")
else:
    print("All files valid.")

Validating 150,017 files...


  0%|          | 0/150017 [00:00<?, ?it/s]

Corrupted/invalid: 3
  17a4969ee8101d2d343cfc1aa9894ae6.npy: bad shape (128, 428)
  720dc2101e81a51e38d7a1524cda0dad.npy: bad shape (128, 418)
  2241cda11073a7004f83d451060c5327.npy: bad shape (128, 438)

Removed 3 file(s).


In [16]:
print(f"{len(list(OUT_DIR.glob('*.npy'))):,} files in {OUT_DIR}")   

150,014 files in /Users/yhkim/GU_work/02_Spring_2026/6600_work/04_PROJ/6600_Project/data/processed_data/piano_roll
